# Stage-1 in NLP Pipeline: Text Preprocessing → Embedding Preparation

This stage converts **raw human text into numerical vectors** that a transformer model can process.

Pipeline:

Text → Tokens → Token IDs → Token Embeddings → Positional Embeddings → Transformer Input

The goal is to transform language into a **matrix representation** that neural networks can understand.

---

# 1. Import Libraries

The system requires three main libraries.

### PyTorch
Used for tensor computation and neural network layers.

PyTorch provides:
- tensors
- neural network modules
- automatic differentiation

Example tensor:

```

tensor([1,2,3])

```

### torch.nn
Contains neural network building blocks such as:

- Embedding layers
- Linear layers
- Layer normalization

### tiktoken
Tokenizer used for GPT-2 style Byte Pair Encoding (BPE).

It converts text into tokens based on a predefined vocabulary.

GPT-2 vocabulary size:

```

50257 tokens

```

Tokens can represent:

- full words
- sub-words
- punctuation
- whitespace

Example tokens:

```

"hello"
" world"
"ing"
"."

```

Notice the space before **world**.

---

# 2. Tokenization

Tokenization converts **raw text into tokens**.

Example input text:

```

Transformers are powerful models

```

After tokenization:

```

["Transformers", " are", " powerful", " models"]

```

The tokenizer splits text based on learned subword patterns.

---

# 3. Token IDs

Each token is mapped to a **unique integer index** from the vocabulary.

Example:

```

Transformers → 41762
are         → 389
powerful    → 4096
models      → 4981

```

The sentence becomes:

```

[41762, 389, 4096, 4981]

```

Neural networks cannot process text directly, so tokens must be represented numerically.

These integers are then converted into tensors.

Example tensor:

```

tensor([41762, 389, 4096, 4981])

```

Shape:

```

(4)

```

Meaning there are **4 tokens in the sequence**.

---

# 4. Token Embedding

Token IDs do not carry semantic meaning.

For example:

```

dog = 5234
cat = 8321

```

The numbers themselves have no relationship.

To solve this, tokens are converted into **dense vectors** using an embedding layer.

The embedding layer is a lookup table.

Embedding matrix shape:

```

[vocabulary_size × embedding_dimension]

```

Example:

```

50257 × 768

```

Meaning:

- 50257 tokens in the vocabulary
- each token has a vector of size 768

Example token vector:

```

[0.23, -0.44, 0.98, ..., -0.11]

```

Shape of token embeddings for the example sentence:

```

(4 × 768)

```

Meaning:

```

4 tokens
768 dimensional vector representation per token

```

This vector representation encodes semantic information learned during training.

---

# 5. Positional Embedding

Transformers process tokens **in parallel**, unlike RNNs or LSTMs.

Because of this, the model does not automatically know the order of words.

Example sentences:

```

Dog bites man
Man bites dog

```

Both contain the same words but have different meanings.

To preserve word order, we introduce **positional embeddings**.

Each position in the sequence gets a unique vector.

Position examples:

```

position 0
position 1
position 2
position 3

```

Positional embedding matrix shape:

```

[max_sequence_length × embedding_dimension]

```

Example:

```

1024 × 768

```

Meaning the model can handle sequences up to **1024 tokens**.

Each position has a 768-dimensional vector.

Example position vector:

```

[0.19, 0.42, -0.12, ..., 0.77]

```

---

# 6. Position Indices

For a sequence of length 4, we generate indices:

```

[0,1,2,3]

```

These represent token positions in the sentence.

Converted into a tensor:

```

tensor([0,1,2,3])

```

---

# 7. Combining Token and Position Embeddings

The final transformer input is created by adding the token vectors and position vectors.

Formula:

```

TransformerInput = TokenEmbedding + PositionEmbedding

```

Example:

Token vector:

```

[0.34, -0.88, 0.72]

```

Position vector:

```

[0.19, 0.42, -0.12]

```

Result:

```

[0.53, -0.46, 0.60]

```

The addition is element-wise.

---

# 8. Final Transformer Input

After combining embeddings, the final tensor shape becomes:

```

(sequence_length × embedding_dimension)

```

Example:

```

(4 × 768)

```

Meaning:

- 4 tokens
- each represented by a 768-dimensional vector

This matrix becomes the **input to the transformer layers**.

---

# 9. Complete Stage-1 Pipeline

```

Raw Text
↓
Tokenization (BPE)
↓
Token IDs
↓
Token Embedding
↓
Positional Embedding
↓
Add both embeddings
↓
Transformer Input Matrix

```

---

# 10. Shape Flow Example

Input sentence:

```

Transformers are powerful models

```

Processing shapes:

```

Token IDs
(4)

Token Embeddings
(4 × 768)

Position Embeddings
(4 × 768)

Final Transformer Input
(4 × 768)

```

---

# 11. Why This Stage Is Important

Neural networks cannot interpret language directly.

Computers cannot understand:

```

"Transformers are powerful"

```

They must operate on numbers.

So the preprocessing stage converts language into a **structured numeric representation** suitable for deep learning.

This representation is then passed to the transformer architecture for attention computation.

---

# 12. What Happens Next

After Stage-1, the next stage of the transformer begins.

The input matrix is used to compute:

```

Query
Key
Value

```

These are used inside **self-attention**, which allows the model to learn relationships between words in a sequence.



In [1]:
import torch
import torch.nn as nn
import tiktoken


# -----------------------------
# 1. Tokenization
# -----------------------------
def tokenize_text(input_text):

    tokenizer = tiktoken.get_encoding("gpt2")

    token_ids = tokenizer.encode(input_text)

    token_ids_tensor = torch.tensor(token_ids)

    return token_ids_tensor, tokenizer


# -----------------------------
# 2. Token Embedding Layer
# -----------------------------
def create_token_embedding_layer(vocabulary_size, embedding_dimension):

    token_embedding_layer = nn.Embedding(
        num_embeddings=vocabulary_size,
        embedding_dim=embedding_dimension
    )

    return token_embedding_layer


# -----------------------------
# 3. Positional Embedding Layer
# -----------------------------
def create_position_embedding_layer(maximum_sequence_length, embedding_dimension):

    position_embedding_layer = nn.Embedding(
        num_embeddings=maximum_sequence_length,
        embedding_dim=embedding_dimension
    )

    return position_embedding_layer


# -----------------------------
# 4. Generate Position IDs
# -----------------------------
def generate_position_indices(sequence_length):

    position_indices = torch.arange(sequence_length)

    return position_indices


# -----------------------------
# 5. Build Transformer Input
# -----------------------------
def build_transformer_input(token_ids,
                            token_embedding_layer,
                            position_embedding_layer):

    sequence_length = token_ids.size(0)

    # Token embeddings
    token_vectors = token_embedding_layer(token_ids)

    # Position indices
    position_indices = generate_position_indices(sequence_length)

    # Position embeddings
    position_vectors = position_embedding_layer(position_indices)

    # Final transformer input
    transformer_input = token_vectors + position_vectors

    return token_vectors, position_vectors, transformer_input


# -----------------------------
# Main Execution
# -----------------------------
def main():

    input_text = "Transformers are powerful models"

    token_ids, tokenizer = tokenize_text(input_text)

    print("\nToken IDs:")
    print(token_ids)

    vocabulary_size = tokenizer.n_vocab
    embedding_dimension = 768
    maximum_sequence_length = 1024

    token_embedding_layer = create_token_embedding_layer(
        vocabulary_size,
        embedding_dimension
    )

    position_embedding_layer = create_position_embedding_layer(
        maximum_sequence_length,
        embedding_dimension
    )

    token_vectors, position_vectors, transformer_input = build_transformer_input(
        token_ids,
        token_embedding_layer,
        position_embedding_layer
    )

    print("\nToken Embedding Vectors Shape:")
    print(token_vectors.shape)

    print("\nToken Embedding Vectors:")
    print(token_vectors)

    print("\nPosition Embedding Vectors:")
    print(position_vectors)

    print("\nFinal Transformer Input (Token + Position):")
    print(transformer_input)

    print("\nFinal Input Shape:")
    print(transformer_input.shape)


if __name__ == "__main__":
    main()


Token IDs:
tensor([41762,   364,   389,  3665,  4981])

Token Embedding Vectors Shape:
torch.Size([5, 768])

Token Embedding Vectors:
tensor([[-1.9591e-01,  8.2297e-01,  5.6309e-01,  ...,  1.2056e+00,
         -1.0494e+00,  2.7096e-01],
        [ 3.5483e+00,  1.3058e+00, -1.9060e-01,  ..., -7.3882e-01,
          1.6129e-02, -9.9488e-02],
        [ 1.1084e-02, -8.4871e-01, -2.6317e-01,  ...,  8.1419e-01,
          1.5924e-01, -5.8029e-01],
        [-1.3695e+00, -8.3195e-01,  6.2356e-02,  ..., -6.5580e-01,
          1.1741e+00, -5.7454e-01],
        [-7.0057e-01,  4.7446e-01,  1.8838e+00,  ...,  3.1614e-03,
          1.1734e-01, -6.7731e-01]], grad_fn=<EmbeddingBackward0>)

Position Embedding Vectors:
tensor([[-1.7241, -0.8541, -0.0998,  ...,  1.1438, -0.4195,  1.6305],
        [-0.0823,  0.7510, -0.6257,  ...,  0.4205,  1.5046,  0.8629],
        [ 1.5788,  0.6082, -0.0327,  ...,  0.1372, -0.6735,  0.3821],
        [ 1.2497, -0.0509, -0.3644,  ...,  0.4264,  0.5348,  1.2323],
        [-